<a href="https://colab.research.google.com/github/sravanthmouli/Not_Every_Agent_Needs_a_Frontier_Model/blob/main/Notebook_CYPHER2026_Not_Every_Agent_Needs_a_Frontier_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 CYPHER 2026 - Workshop

**Google Colab / Local Ready** | Qwen2.5-0.5B (SLM) + Qwen3-8B (LLM)

### Stage Classification

| Stage | Model | Why |
|-------|-------|-----|
| Keyword extraction / query rewrite | **SLM** | Structured, low ambiguity |
| Paper validation / filtering | **SLM** | Classification |
| Full research planning | **LLM** | Open-ended synthesis |
| AGI multi-criteria evaluation | **LLM** | Nuanced judgment |
| Final report synthesis | **LLM** | Long-form writing |

## Workshop map (read this first)

| Goal | Where in this notebook |
|------|------------------------|
| Stage → model policy | §7 `choose_model()` |
| SLM vs LLM vs Hybrid run | §10 |
| Metrics (quality, latency, cost, call mix) | §11 dashboard |
| Discussion prompts | §13 |

**Models:** SLM = `Qwen2.5-0.5B-Instruct` · LLM = `Qwen3-8B` (4-bit on GPU)

**Three modes**
- **SLM** — every stage uses the 0.5B model  
- **LLM** — every stage uses the 8B model  
- **Hybrid** — structured stages → SLM; judgment / synthesis → LLM


## 1. Installation

Run once, then **Runtime → Restart session**.

In [1]:
!pip install -q transformers>=4.46.0 accelerate bitsandbytes sentencepiece
!pip install -q langchain==0.3.14 langchain-core==0.3.29 langgraph==0.2.60
!pip install -q arxiv==2.1.3 tenacity==9.0.0 python-dotenv pandas
print("✅ Installation complete. Please Runtime → Restart session now.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.29 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the p

## 2. Configuration

In [2]:
import os, json, time, uuid, logging, re, statistics, warnings, random
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, TypedDict, Literal
from dataclasses import dataclass
from enum import Enum
from collections import defaultdict
warnings.filterwarnings("ignore")

# ------------------------------
# MODEL CONFIGURATION
# ------------------------------
SLM_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
LLM_MODEL_ID = "Qwen/Qwen3-8B"

DEFAULT_MODE = "hybrid"
USE_4BIT = True
MAX_NEW_TOKENS_SLM = 512
MAX_NEW_TOKENS_LLM = 1024
TEMPERATURE = 0.3

# Research defaults
MAX_PAPERS_DEFAULT = 3
RESEARCH_DAYS_LOOKBACK = 30          # widened from 14 → more robust
ARXIV_MAX_RETRIES = 5
ARXIV_BASE_DELAY = 3.0               # seconds

print(f"SLM : {SLM_MODEL_ID}")
print(f"LLM : {LLM_MODEL_ID} (4-bit={USE_4BIT})")
print(f"Lookback days: {RESEARCH_DAYS_LOOKBACK}")

SLM : Qwen/Qwen2.5-0.5B-Instruct
LLM : Qwen/Qwen3-8B (4-bit=True)
Lookback days: 30


## 3. Model Manager

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class ModelManager:
    """Unified SLM/LLM interface with metrics."""

    def __init__(self, slm_id=SLM_MODEL_ID, llm_id=LLM_MODEL_ID, use_4bit=USE_4BIT):
        self.slm_id, self.llm_id, self.use_4bit = slm_id, llm_id, use_4bit
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.slm = self.llm = self.slm_tokenizer = self.llm_tokenizer = None
        self.metrics = {
            "slm_calls": 0, "llm_calls": 0,
            "slm_tokens_in": 0, "slm_tokens_out": 0,
            "llm_tokens_in": 0, "llm_tokens_out": 0,
            "slm_latencies": [], "llm_latencies": [],
        }
        print(f"Device: {self.device}")
        if self.device == "cuda":
            print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    def _bnb(self):
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

    def load_slm(self):
        if self.slm is not None: return
        print(f"⏳ Loading SLM: {self.slm_id} ...")
        t0 = time.time()
        self.slm_tokenizer = AutoTokenizer.from_pretrained(self.slm_id, trust_remote_code=True)
        self.slm = AutoModelForCausalLM.from_pretrained(
            self.slm_id, torch_dtype=torch.bfloat16 if self.device=="cuda" else torch.float32,
            device_map="auto", trust_remote_code=True)
        self.slm.eval()
        print(f"✅ SLM loaded in {time.time()-t0:.1f}s")

    def load_llm(self):
        if self.llm is not None: return
        print(f"⏳ Loading LLM: {self.llm_id} (4-bit={self.use_4bit}) ...")
        t0 = time.time()
        self.llm_tokenizer = AutoTokenizer.from_pretrained(self.llm_id, trust_remote_code=True)
        kwargs = dict(device_map="auto", trust_remote_code=True)
        if self.use_4bit and self.device == "cuda":
            kwargs["quantization_config"] = self._bnb()
        else:
            kwargs["torch_dtype"] = torch.bfloat16 if self.device=="cuda" else torch.float32
        self.llm = AutoModelForCausalLM.from_pretrained(self.llm_id, **kwargs)
        self.llm.eval()
        print(f"✅ LLM loaded in {time.time()-t0:.1f}s")

    def load_all(self, mode="hybrid"):
        if mode in ("slm", "hybrid"): self.load_slm()
        if mode in ("llm", "hybrid"): self.load_llm()

    def _chat(self, tokenizer, system, user):
        msgs = [{"role":"system","content":system},{"role":"user","content":user}]
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    @torch.inference_mode()
    def generate(self, prompt, model_type="slm", system="You are a helpful research assistant.",
                 max_new_tokens=None, temperature=TEMPERATURE):
        if model_type == "slm":
            if self.slm is None: self.load_slm()
            model, tok = self.slm, self.slm_tokenizer
            max_new_tokens = max_new_tokens or MAX_NEW_TOKENS_SLM
        else:
            if self.llm is None: self.load_llm()
            model, tok = self.llm, self.llm_tokenizer
            max_new_tokens = max_new_tokens or MAX_NEW_TOKENS_LLM

        full = self._chat(tok, system, prompt)
        inputs = tok(full, return_tensors="pt").to(model.device)
        in_len = inputs["input_ids"].shape[1]

        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                             do_sample=temperature>0, top_p=0.9, pad_token_id=tok.eos_token_id)
        latency = time.time() - t0
        gen = out[0][in_len:]
        text = tok.decode(gen, skip_special_tokens=True).strip()
        out_len = len(gen)

        prefix = "slm" if model_type=="slm" else "llm"
        self.metrics[f"{prefix}_calls"] += 1
        self.metrics[f"{prefix}_tokens_in"] += in_len
        self.metrics[f"{prefix}_tokens_out"] += out_len
        self.metrics[f"{prefix}_latencies"].append(latency)
        return text

    def reset_metrics(self):
        for k in self.metrics:
            self.metrics[k] = [] if isinstance(self.metrics[k], list) else 0

    def get_metrics_summary(self):
        def pct(xs, p):
            if not xs: return 0.0
            if len(xs) == 1: return xs[0]
            return float(statistics.quantiles(xs, n=100)[p-1])
        return {
            "slm_calls": self.metrics["slm_calls"],
            "llm_calls": self.metrics["llm_calls"],
            "total_calls": self.metrics["slm_calls"] + self.metrics["llm_calls"],
            "slm_tokens": self.metrics["slm_tokens_in"] + self.metrics["slm_tokens_out"],
            "llm_tokens": self.metrics["llm_tokens_in"] + self.metrics["llm_tokens_out"],
            "slm_latency_p50": pct(self.metrics["slm_latencies"], 50),
            "slm_latency_p95": pct(self.metrics["slm_latencies"], 95),
            "llm_latency_p50": pct(self.metrics["llm_latencies"], 50),
            "llm_latency_p95": pct(self.metrics["llm_latencies"], 95),
            "total_latency": sum(self.metrics["slm_latencies"]) + sum(self.metrics["llm_latencies"]),
        }

model_manager = None
print("ModelManager ready.")

ModelManager ready.


## 4. Logging & State

In [6]:
class ResearchPhase(Enum):
    INITIALIZATION = "initialization"
    PLANNING = "planning"
    DISCOVERY = "discovery"
    EVALUATION = "evaluation"
    COMPLETION = "completion"

class ResearchSystemState(TypedDict, total=False):
    request_id: str
    research_objective: str
    current_phase: ResearchPhase
    phase_history: List[str]
    execution_plan: Dict[str, Any]
    discovered_papers: List[Dict[str, Any]]
    evaluation_results: List[Dict[str, Any]]
    final_report: str
    mode: str
    metrics: Dict[str, Any]
    created_at: str
    updated_at: str
    error: Optional[str]

class ResearchLogger:
    def __init__(self, name):
        self.logger = logging.getLogger(name)
        if not self.logger.handlers:
            h = logging.StreamHandler()
            h.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
            self.logger.addHandler(h)
            self.logger.setLevel(logging.INFO)
    def info(self, msg, **kw):
        extra = " | ".join(f"{k}={v}" for k,v in kw.items()) if kw else ""
        self.logger.info(f"{msg} {extra}".strip())

def get_logger(name): return ResearchLogger(name)
print("State & logging ready.")

State & logging ready.


## 5. ArXiv Search (FIXED – retries + fallback)

In [7]:
import arxiv
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

class ArxivRateLimitError(Exception):
    pass

def _arxiv_search_once(query: str, max_papers: int, from_date: str, to_date: str) -> List[Dict]:
    """Single attempt. Raises ArxivRateLimitError on 429."""
    date_filter = ""
    if from_date and to_date:
        fd, td = from_date.replace("-",""), to_date.replace("-","")
        date_filter = f" AND submittedDate:[{fd}0000 TO {td}2359]"

    full_query = f"({query}) AND (cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR cat:cs.CV OR cat:cs.NE){date_filter}"
    logger = get_logger("search_arxiv")
    logger.info("ArXiv query", query=full_query[:200])

    try:
        search = arxiv.Search(
            query=full_query,
            max_results=max_papers,
            sort_by=arxiv.SortCriterion.SubmittedDate,
            sort_order=arxiv.SortOrder.Descending,
        )
        results = list(search.results())
    except Exception as e:
        err = str(e).lower()
        if "429" in err or "rate" in err or "too many" in err:
            raise ArxivRateLimitError(str(e))
        logger.info(f"ArXiv error (non-429): {e}")
        return []

    papers = []
    for r in results:
        papers.append({
            "title": r.title,
            "authors": [a.name for a in r.authors],
            "abstract": r.summary.replace("\n", " ").strip(),
            "published": r.published.isoformat() if r.published else None,
            "pdf_url": r.pdf_url,
            "entry_id": r.entry_id,
            "categories": r.categories,
            "source": "arxiv",
        })
    return papers


def search_arxiv(query: str, max_papers: int = 10,
                 from_date: str = None, to_date: str = None) -> List[Dict]:
    """
    Robust ArXiv search with exponential backoff on rate limits.
    """
    logger = get_logger("search_arxiv")
    delay = ARXIV_BASE_DELAY

    for attempt in range(1, ARXIV_MAX_RETRIES + 1):
        try:
            papers = _arxiv_search_once(query, max_papers, from_date, to_date)
            logger.info("ArXiv success", papers=len(papers), attempt=attempt)
            return papers
        except ArxivRateLimitError as e:
            if attempt == ARXIV_MAX_RETRIES:
                logger.info(f"ArXiv rate-limit: giving up after {attempt} attempts")
                return []
            sleep_for = delay + random.uniform(0, 2)
            logger.info(f"ArXiv 429 – sleeping {sleep_for:.1f}s (attempt {attempt}/{ARXIV_MAX_RETRIES})")
            time.sleep(sleep_for)
            delay *= 2
    return []


# Hard fallback query that almost always returns results
FALLBACK_QUERY = '("artificial general intelligence" OR AGI OR "general-purpose agent" OR "reasoning agent")'

print("ArXiv tool (with retries) ready.")

ArXiv tool (with retries) ready.


## 6. AGI Evaluation Prompt & Robust Parser

In [8]:
def get_agi_evaluation_prompt(title: str, abstract: str, authors: List[str]) -> str:
    authors_str = ", ".join(authors[:5])
    return f"""EVALUATE THIS RESEARCH PAPER FOR AGI POTENTIAL.

Title: {title}
Authors: {authors_str}
Abstract: {abstract[:1200]}

Score each parameter 1-10 (1-3 minimal, 4-6 some, 7-8 strong, 9-10 exceptional):
1. Novel Problem Solving (15%)
2. Few-Shot Learning (15%)
3. Task Transfer (15%)
4. Abstract Reasoning (12%)
5. Contextual Adaptation (10%)
6. Multi-Rule Integration (10%)
7. Generalization Efficiency (8%)
8. Meta-Learning (8%)
9. World Modeling (4%)
10. Autonomous Goal Setting (3%)

Reply with ONLY valid JSON (no markdown, no thinking):
{{
  "scores": {{
    "novel_problem_solving": <int>,
    "few_shot_learning": <int>,
    "task_transfer": <int>,
    "abstract_reasoning": <int>,
    "contextual_adaptation": <int>,
    "multi_rule_integration": <int>,
    "generalization_efficiency": <int>,
    "meta_learning": <int>,
    "world_modeling": <int>,
    "autonomous_goal_setting": <int>
  }},
  "overall_assessment": "<1-2 sentences>",
  "key_innovations": ["<item1>", "<item2>"]
}}"""

WEIGHTS = {
    "novel_problem_solving": 0.15, "few_shot_learning": 0.15, "task_transfer": 0.15,
    "abstract_reasoning": 0.12, "contextual_adaptation": 0.10, "multi_rule_integration": 0.10,
    "generalization_efficiency": 0.08, "meta_learning": 0.08,
    "world_modeling": 0.04, "autonomous_goal_setting": 0.03,
}

def _extract_json(text: str) -> Optional[dict]:
    """Try multiple strategies to extract a JSON object."""
    # Strip common thinking tags / markdown
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL|re.I)
    cleaned = re.sub(r"```json\s*", "", cleaned)
    cleaned = re.sub(r"```\s*", "", cleaned)

    # Strategy 1: direct parse
    try:
        return json.loads(cleaned.strip())
    except: pass

    # Strategy 2: find outermost { ... }
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except: pass

    # Strategy 3: find "scores" block and rebuild
    scores_match = re.search(r'"scores"\s*:\s*\{([^}]+)\}', cleaned, re.DOTALL)
    if scores_match:
        try:
            scores = {}
            for k in WEIGHTS:
                m = re.search(rf'"{k}"\s*:\s*(\d+)', scores_match.group(1))
                if m: scores[k] = int(m.group(1))
            if scores:
                return {"scores": scores, "overall_assessment": "Partial parse", "key_innovations": []}
        except: pass
    return None


def parse_agi_response(text: str) -> Dict[str, Any]:
    data = _extract_json(text)
    if data is None:
        # Last resort: look for any integers that might be scores
        nums = [int(x) for x in re.findall(r'\b([1-9]|10)\b', text)]
        if len(nums) >= 5:
            # Use first 10 numbers as rough scores
            keys = list(WEIGHTS.keys())
            scores = {keys[i]: nums[i] for i in range(min(10, len(nums)))}
            data = {"scores": scores, "overall_assessment": "Recovered from numbers", "key_innovations": []}
        else:
            return {"agi_score": 30.0, "agi_classification": "Low AGI Potential (parse fallback)",
                    "scores": {}, "overall_assessment": "Could not parse model output",
                    "key_innovations": [], "parse_failed": True}

    scores = data.get("scores", {})
    weighted = sum(float(scores.get(k, 5)) * w for k, w in WEIGHTS.items())  # default 5 if missing
    agi_score = round(weighted * 10, 1)

    if agi_score >= 70: classification = "High AGI Potential"
    elif agi_score >= 40: classification = "Medium AGI Potential"
    else: classification = "Low AGI Potential"

    return {
        "agi_score": agi_score,
        "agi_classification": classification,
        "scores": scores,
        "overall_assessment": data.get("overall_assessment", ""),
        "key_innovations": data.get("key_innovations", []),
        "parse_failed": False,
    }

print("AGI helpers (robust parser) ready.")

AGI helpers (robust parser) ready.


## 7. Hybrid Agents (FIXED query rewrite + fallbacks)

In [9]:
def choose_model(stage: str, mode: str) -> Literal["slm", "llm"]:
    """
    Explicit per-stage routing (this is the core of the workshop).

    SLM-appropriate (cheap / structured):
      keyword_extraction, query_rewrite, routing, paper_validation

    LLM-required (judgment / synthesis):
      agi_evaluation, report_synthesis, open-ended planning

    Modes:
      slm    → force SLM for every stage
      llm    → force LLM for every stage
      hybrid → use the mapping above
    """
    if mode == "slm": return "slm"
    if mode == "llm": return "llm"
    slm_stages = {"keyword_extraction", "query_rewrite", "paper_validation", "routing"}
    return "slm" if stage in slm_stages else "llm"


def _is_bad_query(q: str) -> bool:
    """Reject natural-language sentences that make poor ArXiv queries."""
    q = q.strip().lower()
    if len(q) < 5: return True
    bad_starts = ("find ", "search for", "recent papers", "papers on", "look for", "get me")
    if any(q.startswith(b) for b in bad_starts): return True
    if q.count(" ") > 12: return True   # too long / sentence-like
    return False


class PlannerAgent:
    def __init__(self, mm: ModelManager):
        self.mm = mm
        self.logger = get_logger("planner")

    def create_plan(self, objective: str, mode: str) -> Dict[str, Any]:
        self.logger.info("Creating plan", mode=mode)

        # Keyword extraction (SLM-friendly)
        kw_model = choose_model("keyword_extraction", mode)
        kw_prompt = f"""List 6-10 academic search keywords for this research goal.
Return ONLY a JSON array of short phrases (no sentences).

Goal: {objective}

Example: ["artificial general intelligence", "AGI", "reasoning agents", "transfer learning"]"""
        kw_raw = self.mm.generate(kw_prompt, model_type=kw_model,
                                  system="You output only JSON arrays of keywords.")
        try:
            keywords = json.loads(re.search(r"\[.*\]", kw_raw, re.DOTALL).group())
            keywords = [k for k in keywords if isinstance(k, str) and len(k) < 60][:12]
        except:
            keywords = ["Artificial General Intelligence", "AGI", "general-purpose agent",
                        "reasoning", "transfer learning", "foundation models"]

        today = datetime.utcnow().strftime("%Y-%m-%d")
        from_date = (datetime.utcnow() - timedelta(days=RESEARCH_DAYS_LOOKBACK)).strftime("%Y-%m-%d")

        # Build a safe default search query from keywords
        safe_query = " OR ".join(f'"{k}"' if " " in k else k for k in keywords[:6])

        plan = {
            "search_keywords": keywords,
            "search_query": safe_query,
            "date_range": {"from": from_date, "to": today},
            "max_papers": MAX_PAPERS_DEFAULT,
            "focus_areas": ["AGI", "reasoning", "generalization"],
            "exclusions": ["narrow AI only"],
            "success_criteria": {"min_papers": 2, "quality_threshold": 0.4},
        }
        self.logger.info("Plan created", keywords=len(keywords))
        return plan


class DiscoveryAgent:
    def __init__(self, mm: ModelManager):
        self.mm = mm
        self.logger = get_logger("discovery")

    def rewrite_query(self, original: str, mode: str) -> str:
        model = choose_model("query_rewrite", mode)
        prompt = f"""Rewrite into a short ArXiv keyword query (max 12 words).
Use OR between terms. Do NOT write a full sentence. Do NOT start with Find/Search/Papers.

Original: {original}

Good example: "artificial general intelligence" OR AGI OR "reasoning agent" OR "transfer learning"
Return ONLY the query string."""
        rewritten = self.mm.generate(prompt, model_type=model,
                                     system="You produce short academic keyword queries only.").strip().strip('"')

        if _is_bad_query(rewritten):
            self.logger.info("Rewrite looked like natural language – using original/safe query")
            return original
        return rewritten

    def discover(self, plan: Dict[str, Any], mode: str) -> List[Dict[str, Any]]:
        query = plan.get("search_query") or FALLBACK_QUERY
        query = self.rewrite_query(query, mode)

        date_range = plan.get("date_range", {})
        max_papers = plan.get("max_papers", MAX_PAPERS_DEFAULT)
        from_d, to_d = date_range.get("from"), date_range.get("to")

        papers = search_arxiv(query, max_papers=max_papers, from_date=from_d, to_date=to_d)

        # Fallback 1: drop date filter
        if not papers:
            self.logger.info("0 papers – retrying without date filter")
            papers = search_arxiv(query, max_papers=max_papers, from_date=None, to_date=None)

        # Fallback 2: hard-coded high-recall query
        if not papers:
            self.logger.info("Still 0 papers – using FALLBACK_QUERY")
            papers = search_arxiv(FALLBACK_QUERY, max_papers=max_papers, from_date=from_d, to_date=to_d)
            if not papers:
                papers = search_arxiv(FALLBACK_QUERY, max_papers=max_papers)

        self.logger.info("Discovery finished", papers=len(papers))
        return papers


class EvaluationAgent:
    def __init__(self, mm: ModelManager):
        self.mm = mm
        self.logger = get_logger("evaluation")

    def evaluate_paper(self, paper: Dict, mode: str) -> Dict:
        model = choose_model("agi_evaluation", mode)
        prompt = get_agi_evaluation_prompt(paper.get("title",""), paper.get("abstract",""), paper.get("authors",[]))
        raw = self.mm.generate(prompt, model_type=model,
                               system="You are an AGI researcher. Output ONLY valid JSON. No markdown, no thinking.")
        result = parse_agi_response(raw)
        result.update({
            "paper_title": paper.get("title"),
            "paper_authors": paper.get("authors", []),
            "published": paper.get("published"),
            "source": paper.get("source"),
            "model_used": model,
        })
        return result

    def evaluate_all(self, papers: List[Dict], mode: str) -> List[Dict]:
        results = []
        for i, p in enumerate(papers, 1):
            self.logger.info(f"Evaluating {i}/{len(papers)}", title=p.get("title","")[:55])
            results.append(self.evaluate_paper(p, mode))
            # Small pause to be kind to GPU / avoid thermal throttling
            time.sleep(0.5)
        return results


class ReportAgent:
    def __init__(self, mm: ModelManager):
        self.mm = mm

    def generate_report(self, state: Dict, mode: str) -> str:
        model = choose_model("report_synthesis", mode)
        evals = state.get("evaluation_results", [])
        if not evals:
            return "No evaluation results available (discovery returned 0 papers)."

        sorted_evals = sorted(evals, key=lambda x: x.get("agi_score", 0), reverse=True)
        lines = [f"{i}. {e.get('paper_title','?')[:80]} — {e.get('agi_score',0)}/100 ({e.get('agi_classification')})"
                 for i, e in enumerate(sorted_evals[:5], 1)]

        prompt = f"""Write a short executive research report in markdown (under 350 words).

Objective: {state.get('research_objective')}
Mode: {mode}
Papers evaluated: {len(evals)}
Average AGI score: {sum(e.get('agi_score',0) for e in evals)/len(evals):.1f}

Top papers:
{chr(10).join(lines)}

Sections: Executive Summary (3-4 sentences), Key Findings, Recommendations.
Output the report only – no thinking tags."""
        return self.mm.generate(prompt, model_type=model,
                                system="You write clear, concise executive research summaries.")

print("Hybrid agents (fixed) ready.")

Hybrid agents (fixed) ready.


## 8. Orchestrator

In [10]:
def run_research(objective: str, mode: str = "hybrid",
                 max_papers: int = MAX_PAPERS_DEFAULT,
                 mm: ModelManager = None) -> Dict[str, Any]:
    global model_manager
    if mm is None:
        if model_manager is None:
            model_manager = ModelManager()
            model_manager.load_all(mode)
        mm = model_manager

    mm.reset_metrics()
    request_id = str(uuid.uuid4())[:8]
    t_start = time.time()

    state: ResearchSystemState = {
        "request_id": request_id,
        "research_objective": objective,
        "current_phase": ResearchPhase.INITIALIZATION,
        "phase_history": ["initialization"],
        "mode": mode,
        "discovered_papers": [],
        "evaluation_results": [],
        "created_at": datetime.utcnow().isoformat(),
    }

    print(f"\n{'='*70}")
    print(f"🚀 mode={mode.upper()} | id={request_id}")
    print(f"Objective: {objective}")
    print(f"{'='*70}")

    try:
        # PLANNING
        state["current_phase"] = ResearchPhase.PLANNING
        state["phase_history"].append("planning")
        plan = PlannerAgent(mm).create_plan(objective, mode)
        plan["max_papers"] = max_papers
        state["execution_plan"] = plan
        print(f"✅ Planning | keywords={len(plan.get('search_keywords',[]))}")

        # DISCOVERY
        state["current_phase"] = ResearchPhase.DISCOVERY
        state["phase_history"].append("discovery")
        papers = DiscoveryAgent(mm).discover(plan, mode)
        state["discovered_papers"] = papers
        print(f"✅ Discovery | papers={len(papers)}")

        if not papers:
            state["final_report"] = "No papers found even after fallbacks. Try a broader query or larger lookback."
            state["current_phase"] = ResearchPhase.COMPLETION
            metrics = mm.get_metrics_summary()
            metrics["wall_clock_sec"] = round(time.time() - t_start, 2)
            metrics.update({"avg_agi_score": 0, "high_count": 0, "medium_count": 0,
                            "papers_discovered": 0, "papers_evaluated": 0})
            state["metrics"] = metrics
            return state

        # EVALUATION
        state["current_phase"] = ResearchPhase.EVALUATION
        state["phase_history"].append("evaluation")
        evaluations = EvaluationAgent(mm).evaluate_all(papers, mode)
        state["evaluation_results"] = evaluations

        avg = sum(e.get("agi_score",0) for e in evaluations) / len(evaluations)
        high = sum(1 for e in evaluations if "High" in e.get("agi_classification",""))
        medium = sum(1 for e in evaluations if "Medium" in e.get("agi_classification",""))
        print(f"✅ Evaluation | avg={avg:.1f} | high={high} medium={medium}")

        # REPORT
        state["current_phase"] = ResearchPhase.COMPLETION
        state["phase_history"].append("completion")
        state["final_report"] = ReportAgent(mm).generate_report(state, mode)

        metrics = mm.get_metrics_summary()
        metrics["wall_clock_sec"] = round(time.time() - t_start, 2)
        metrics.update({
            "avg_agi_score": round(avg, 1),
            "high_count": high, "medium_count": medium,
            "papers_discovered": len(papers),
            "papers_evaluated": len(evaluations),
        })
        state["metrics"] = metrics
        state["updated_at"] = datetime.utcnow().isoformat()

        print(f"\n🏁 Done in {metrics['wall_clock_sec']}s | SLM={metrics['slm_calls']} LLM={metrics['llm_calls']}")
        return state

    except Exception as e:
        get_logger("runner").info(f"Failed: {e}")
        state["error"] = str(e)
        state["metrics"] = mm.get_metrics_summary()
        state["metrics"]["wall_clock_sec"] = round(time.time() - t_start, 2)
        return state

print("Orchestrator ready.")

Orchestrator ready.


## 9. Load Models

In [11]:
model_manager = ModelManager(use_4bit=USE_4BIT)
model_manager.load_all(mode="hybrid")
print("\n✅ Models ready.")

Device: cuda
GPU: Tesla T4 | VRAM: 15.6 GB
⏳ Loading SLM: Qwen/Qwen2.5-0.5B-Instruct ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ LLM loaded in 295.4s

✅ Models ready.


## 9b. Optional smoke test (one mode only)

Use this if you only want a quick check before the full three-mode comparison.
On a T4 this is typically **1–3 minutes** after models are loaded.


In [12]:
# Smoke test — change SMOKE_MODE if needed, then run this cell alone
SMOKE_MODE = "hybrid"   # "slm" | "llm" | "hybrid"
SMOKE_MAX_PAPERS = 2

print(f"Smoke test mode={SMOKE_MODE} max_papers={SMOKE_MAX_PAPERS}")
_smoke = run_research(
    "Find recent papers advancing Artificial General Intelligence or general-purpose reasoning agents",
    mode=SMOKE_MODE,
    max_papers=SMOKE_MAX_PAPERS,
    mm=model_manager,
)
print("Smoke metrics:", _smoke.get("metrics", {}))


2026-10-09 13:14:02,174 - planner - INFO - Creating plan mode=hybrid
INFO:planner:Creating plan mode=hybrid


Smoke test mode=hybrid max_papers=2

🚀 mode=HYBRID | id=4c3480d7
Objective: Find recent papers advancing Artificial General Intelligence or general-purpose reasoning agents


2026-10-09 13:14:05,205 - planner - INFO - Plan created keywords=6
INFO:planner:Plan created keywords=6


✅ Planning | keywords=6


2026-10-09 13:14:05,486 - discovery - INFO - Rewrite looked like natural language – using original/safe query
INFO:discovery:Rewrite looked like natural language – using original/safe query
2026-10-09 13:14:05,489 - search_arxiv - INFO - ArXiv query query=("recent papers" OR "Artificial General Intelligence" OR "general-purpose reasoning agents" OR "machine learning" OR "deep learning" OR "neural networks") AND (cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR c
INFO:search_arxiv:ArXiv query query=("recent papers" OR "Artificial General Intelligence" OR "general-purpose reasoning agents" OR "machine learning" OR "deep learning" OR "neural networks") AND (cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR c
2026-10-09 13:14:05,756 - search_arxiv - INFO - ArXiv success papers=2 | attempt=1
INFO:search_arxiv:ArXiv success papers=2 | attempt=1
2026-10-09 13:14:05,758 - discovery - INFO - Discovery finished papers=2
INFO:discovery:Discovery finished papers=2
2026-10-09 13:14:05,761 - evaluation - INFO - Evaluati

✅ Discovery | papers=2


2026-10-09 13:14:29,137 - evaluation - INFO - Evaluating 2/2 title=A Balanced Data Diet: Addressing the Exploration Bottle
INFO:evaluation:Evaluating 2/2 title=A Balanced Data Diet: Addressing the Exploration Bottle


✅ Evaluation | avg=67.9 | high=1 medium=1

🏁 Done in 88.14s | SLM=2 LLM=3
Smoke metrics: {'slm_calls': 2, 'llm_calls': 3, 'total_calls': 5, 'slm_tokens': 248, 'llm_tokens': 1998, 'slm_latency_p50': 1.6326431035995483, 'slm_latency_p95': 5.29270259141922, 'llm_latency_p50': 26.18943953514099, 'llm_latency_p95': 41.10138912200928, 'total_latency': 86.77716970443726, 'wall_clock_sec': 88.14, 'avg_agi_score': 67.9, 'high_count': 1, 'medium_count': 1, 'papers_discovered': 2, 'papers_evaluated': 2}


## 10. Run All Three Modes

>[Heavy Module - takes time to run] Tip: Between modes we add a short sleep to respect ArXiv rate limits.

In [13]:
RESEARCH_QUERY = "Find recent papers advancing Artificial General Intelligence or general-purpose reasoning agents"
MAX_PAPERS = 3

results = {}

for mode in ["slm", "llm", "hybrid"]:
    print(f"\n\n{'#'*70}\n# MODE: {mode.upper()}\n{'#'*70}")
    state = run_research(RESEARCH_QUERY, mode=mode, max_papers=MAX_PAPERS, mm=model_manager)
    results[mode] = state

    path = f"report_{mode}_{state['request_id']}.md"
    with open(path, "w") as f:
        f.write(state.get("final_report", "No report"))
    print(f"📄 Saved {path}")

    # Be polite to ArXiv between modes
    if mode != "hybrid":
        print("⏳ Cooling down 15s before next mode (ArXiv rate-limit)...")
        time.sleep(15)

print("\n✅ All modes finished.")

2026-10-09 13:15:55,694 - planner - INFO - Creating plan mode=slm
INFO:planner:Creating plan mode=slm




######################################################################
# MODE: SLM
######################################################################

🚀 mode=SLM | id=144f19e3
Objective: Find recent papers advancing Artificial General Intelligence or general-purpose reasoning agents


2026-10-09 13:15:57,689 - planner - INFO - Plan created keywords=6
INFO:planner:Plan created keywords=6


✅ Planning | keywords=6


2026-10-09 13:15:58,257 - discovery - INFO - Rewrite looked like natural language – using original/safe query
INFO:discovery:Rewrite looked like natural language – using original/safe query
2026-10-09 13:15:58,260 - search_arxiv - INFO - ArXiv query query=("recent papers" OR "Artificial General Intelligence" OR "general-purpose reasoning agents" OR "machine learning" OR "deep learning" OR "neural networks") AND (cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR c
INFO:search_arxiv:ArXiv query query=("recent papers" OR "Artificial General Intelligence" OR "general-purpose reasoning agents" OR "machine learning" OR "deep learning" OR "neural networks") AND (cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR c
2026-10-09 13:15:58,453 - search_arxiv - INFO - ArXiv success papers=3 | attempt=1
INFO:search_arxiv:ArXiv success papers=3 | attempt=1
2026-10-09 13:15:58,457 - discovery - INFO - Discovery finished papers=3
INFO:discovery:Discovery finished papers=3
2026-10-09 13:15:58,458 - evaluation - INFO - Evaluati

...

✅ Planning | keywords=6


2026-10-09 13:21:50,098 - discovery - INFO - Rewrite looked like natural language – using original/safe query
INFO:discovery:Rewrite looked like natural language – using original/safe query
2026-10-09 13:21:50,100 - search_arxiv - INFO - ArXiv query query=("recent papers" OR "Artificial General Intelligence" OR "general-purpose reasoning agents" OR "AI research" OR "machine learning" OR "deep learning") AND (cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR cat:c
INFO:search_arxiv:ArXiv query query=("recent papers" OR "Artificial General Intelligence" OR "general-purpose reasoning agents" OR "AI research" OR "machine learning" OR "deep learning") AND (cat:cs.AI OR cat:cs.LG OR cat:cs.CL OR cat:c
2026-10-09 13:21:50,243 - search_arxiv - INFO - ArXiv success papers=3 | attempt=1
INFO:search_arxiv:ArXiv success papers=3 | attempt=1
2026-10-09 13:21:50,244 - discovery - INFO - Discovery finished papers=3
INFO:discovery:Discovery finished papers=3
2026-10-09 13:21:50,247 - evaluation - INFO - Evaluati

✅ Discovery | papers=3


2026-10-09 13:22:15,550 - evaluation - INFO - Evaluating 2/3 title=A Balanced Data Diet: Addressing the Exploration Bottle
INFO:evaluation:Evaluating 2/3 title=A Balanced Data Diet: Addressing the Exploration Bottle
2026-10-09 13:22:41,851 - evaluation - INFO - Evaluating 3/3 title=One Block, Multiple Depths: Recurrent Vision Transforme
INFO:evaluation:Evaluating 3/3 title=One Block, Multiple Depths: Recurrent Vision Transforme


✅ Evaluation | avg=68.1 | high=1 medium=2

🏁 Done in 113.55s | SLM=2 LLM=4
📄 Saved report_hybrid_bc4badb4.md

✅ All modes finished.


## 11. Comparison Dashboard

### How to read these numbers

| Metric | What it answers |
|--------|------------------|
| Avg AGI / High count | Quality of **judgment** (LLM-favored stages) |
| Wall-clock | End-to-end latency |
| SLM vs LLM calls | Did Hybrid actually route, or did everything hit the big model? |
| Est. cost | Token-based proxy (illustrative rates) |

**Fairness note:** In this FIXED run each mode may retrieve a **different paper set** (rewrite + search differ).  
So **latency, tokens, cost, and call mix** are the clean apples-to-apples metrics.  
Avg AGI / High counts are *directional* — useful for discussion, not a pure A/B on the same papers.


In [14]:
import pandas as pd
from IPython.display import display, Markdown

def build_comparison(results):
    rows = []
    for mode, state in results.items():
        m = state.get("metrics", {})
        rows.append({
            "Mode": mode.upper(),
            "Papers": m.get("papers_evaluated", 0),
            "Avg AGI Score": m.get("avg_agi_score", 0),
            "High": m.get("high_count", 0),
            "Medium": m.get("medium_count", 0),
            "Wall-clock (s)": m.get("wall_clock_sec", 0),
            "SLM calls": m.get("slm_calls", 0),
            "LLM calls": m.get("llm_calls", 0),
            "SLM tokens": m.get("slm_tokens", 0),
            "LLM tokens": m.get("llm_tokens", 0),
            "SLM P50 (s)": round(m.get("slm_latency_p50", 0), 2),
            "LLM P50 (s)": round(m.get("llm_latency_p50", 0), 2),
            "SLM P95 (s)": round(m.get("slm_latency_p95", 0), 2),
            "LLM P95 (s)": round(m.get("llm_latency_p95", 0), 2),
        })
    return pd.DataFrame(rows)

df = build_comparison(results)
print("📊 COMPARISON")
display(df)

def est_cost(row):
    return round(row["SLM tokens"]/1e6*0.05 + row["LLM tokens"]/1e6*0.30, 5)

df["Est. Cost ($)"] = df.apply(est_cost, axis=1)
print("\n💰 Cost proxy")
display(df[["Mode","SLM tokens","LLM tokens","Est. Cost ($)"]])

print("\n🏆 TOP PAPER PER MODE")
for mode, state in results.items():
    evals = state.get("evaluation_results", [])
    if evals:
        top = max(evals, key=lambda x: x.get("agi_score", 0))
        print(f"\n[{mode.upper()}] {top.get('paper_title','N/A')[:90]}")
        print(f"     Score: {top.get('agi_score')} | {top.get('agi_classification')} | model={top.get('model_used')}")
    else:
        print(f"\n[{mode.upper()}] No papers evaluated")

📊 COMPARISON


,Mode,Papers,Avg AGI Score,High,Medium,Wall-clock (s),SLM calls,LLM calls,SLM tokens,LLM tokens,SLM P50 (s),LLM P50 (s),SLM P95 (s),LLM P95 (s)
0,SLM,3,68.4,2,1,44.13,6,0,3244,0,5.70,0.00,20.03,0.00
1,LLM,3,66.6,2,1,278.34,0,6,0,4470,0.00,30.63,0.00,168.71
2,HYBRID,3,68.1,1,2,113.55,2,4,251,2804,0.97,25.30,2.18,42.14



💰 Cost proxy


,Mode,SLM tokens,LLM tokens,Est. Cost ($)
0,SLM,3244,0,0.00016
1,LLM,0,4470,0.00134
2,HYBRID,251,2804,0.00085



🏆 TOP PAPER PER MODE

[SLM] One Block, Multiple Depths: Recurrent Vision Transformers with Depth-Programmed Experts
     Score: 79.1 | High AGI Potential | model=slm

[LLM] Scaling to Tens of Thousands of Test-Time Iterations with Loop-Native Attention Residuals
     Score: 70.8 | High AGI Potential | model=llm

[HYBRID] CSF: Contextual Safety Filtering for Motion Generators
     Score: 74.6 | High AGI Potential | model=llm


## 12. Reports

In [15]:
for mode in ["slm", "llm", "hybrid"]:
    state = results.get(mode, {})
    report = state.get("final_report", "No report generated.")
    display(Markdown(f"### Report — {mode.upper()}\n\n{report}\n\n---"))

### Report — SLM

# Executive Research Report

## Objective: Find Recent Papers Advancing Artificial General Intelligence or General-Purpose Reasoning Agents

### Mode: Slm
#### Papers Evaluated: 3
#### Average AGI Score: 68.4

### Top Papers:

1. **One Block, Multiple Depths**: Recurrent Vision Transformers with Depth-Programmed  — 79.1/100 (High AGI Potential)
   - This paper introduces a novel approach to recurrent vision transformers that utilizes depth-programming techniques to improve their performance on tasks like motion generation.

2. **CSF: Contextual Safety Filtering for Motion Generators**: 70.2/100 (High AGI Potential)
   - The authors propose a contextually safe filtering mechanism for motion generators, which is crucial for generating realistic and coherent motion sequences.

3. **A Balanced Data Diet: Addressing the Exploration Bottleneck in Mega-Scale RL for Reinforcement Learning**: 56.0/100 (Medium AGI Potential)
   - This study addresses the challenge of balancing exploration and exploitation in reinforcement learning by proposing a balanced data diet strategy.

### Sections:

**Executive Summary**

Recent papers have significantly advanced artificial intelligence (AI) capabilities, particularly in areas such as general-purpose reasoning agents. This report highlights three notable works that contribute to these advancements.

**Key Findings**

1. **One Block, Multiple Depths**: This paper presents a groundbreaking approach to recurrent vision transformers that leverages depth-programming techniques to enhance their performance on tasks involving motion generation. The authors demonstrate that this method leads to higher AGI scores compared to existing methods.

2. **CSF: Contextual Safety Filtering for Motion Generators**: The authors introduce a new framework called Contextual Safety Filtering (CSF) for motion generators, which aims to ensure that generated motion sequences are safe and reliable. The results show that CSF improves the quality and reliability of motion sequences, making it a promising addition to the field.

3. **A Balanced Data Diet: Addressing the Exploration Bottleneck in Mega-Scale RL for Reinforcement Learning**: This study proposes a novel approach to handling the exploration bottleneck in large-scale reinforcement learning environments. By introducing a balanced data diet strategy, the researchers aim to balance exploration and exploitation more effectively, leading to improved learning outcomes.

**Recommendations**

1. **Expand Research into More Specific Applications**: While the above findings are significant, there may be opportunities to further explore specific applications where these techniques can yield substantial benefits.

2.

---

### Report — LLM

<think>

</think>

# Executive Research Report: Recent Advances in AGI and General-Purpose Reasoning Agents

## Executive Summary  
Recent research highlights promising advancements in AGI and general-purpose reasoning agents. Three papers were evaluated, with an average AGI score of 66.6. Two papers show high AGI potential, while one demonstrates medium potential. These studies focus on iterative learning, self-improvement, and scientific reasoning.

## Key Findings  
1. **Scaling to Tens of Thousands of Test-Time Iterations with Loop-Native Attention** (70.8/100) introduces a novel attention mechanism that enables models to perform extensive iterative reasoning, enhancing their ability to handle complex, sequential tasks.  
2. **Memento 3: Model-Based Recursive Self-Improvement through Reflective Rulebooks** (70.4/100) presents a framework for recursive self-improvement, allowing models to generate and refine their own rulebooks, suggesting a path toward autonomous learning and adaptation.  
3. **DataSense-Bench: The First Step Toward an AI Scientist** (58.5/100) introduces a benchmark for scientific reasoning, showcasing models that can interpret data, form hypotheses, and draw conclusions, marking progress toward AI-driven scientific discovery.

## Recommendations  
Prioritize research on iterative learning and self-improvement frameworks, as they show the highest AGI potential. Additionally, invest in benchmarking tools that evaluate reasoning and scientific understanding to guide the development of more capable general-purpose agents.

---

### Report — HYBRID

<think>
</think>

# Executive Research Report: Recent Advances in AGI and General-Purpose Reasoning Agents

## Executive Summary  
Recent research has made strides in advancing Artificial General Intelligence (AGI) and general-purpose reasoning agents. Three key papers were evaluated, with an average AGI score of 68.1. The top paper, "CSF: Contextual Safety Filtering for Motion Generators," demonstrated the highest potential for AGI development. The other two papers showed medium potential but highlighted important challenges in exploration and architectural design.

## Key Findings  
1. **CSF: Contextual Safety Filtering for Motion Generators** (74.6/100) introduces a novel safety mechanism that enhances the reliability of autonomous systems, suggesting potential for broader AGI applications.  
2. **A Balanced Data Diet: Addressing the Exploration Bottleneck in Mega-Scale RL** (66.6/100) focuses on improving exploration efficiency in reinforcement learning, which is critical for developing adaptable agents.  
3. **One Block, Multiple Depths: Recurrent Vision Transformers with Depth-Programmed** (63.1/100) explores novel transformer architectures for vision tasks, indicating potential for more flexible and generalizable models.

## Recommendations  
- Prioritize research on safety mechanisms and exploration strategies to enhance AGI robustness.  
- Invest in architectural innovations that support both perception and reasoning capabilities.  
- Monitor progress in hybrid models that integrate safety, exploration, and adaptability for long-term AGI development.

---

## 13. Analysis checklist & discussion

Use the dashboard above. Aim for evidence, not vibes.

1. **Quality** — Did LLM/Hybrid surface stronger AGI scores or more “High” papers than pure SLM?
2. **Latency** — Wall-clock ratio Hybrid vs pure LLM? SLM vs LLM?
3. **Call mix** — In Hybrid, what share of calls/tokens went to the SLM?
4. **When is Hybrid worth it?** High-volume monitoring → Hybrid; one-off deep analysis → pure LLM is fine.

### Discussion prompts (5 min)

1. On *this* run, was Hybrid worth it vs pure LLM (latency & cost vs quality)?
2. Which stages would you move to SLM next in your own stack?
3. When would you *force* LLM for planning or rewrite anyway?
4. What would change if we froze the paper list and only varied the evaluator?

> **Takeaway:** Don’t default to the biggest model for every stage. Classification, extraction, and simple rewriting are SLM jobs. Judgment and long-form synthesis still need an LLM.


---
**Files written:** `report_slm_*.md`, `report_llm_*.md`, `report_hybrid_*.md`
